In [1]:
%%writefile plans.json
[
 {
  "plan_id": "P101",
  "plan_name": "Smart Basic",
  "monthly_fee": 499,
  "data_limit_gb": 50,
  "features": {
   "unlimited_calls": true,
   "ott_included": false,
   "roaming": "National"
  }
 },
 {
  "plan_id": "P102",
  "plan_name": "Smart Plus",
  "monthly_fee": 799,
  "data_limit_gb": 75,
  "features": {
   "unlimited_calls": true,
   "ott_included": true,
   "roaming": "National"
  }
 },
 {
  "plan_id": "P103",
  "plan_name": "Budget Saver",
  "monthly_fee": 299,
  "data_limit_gb": 25,
  "features": {
   "unlimited_calls": false,
   "ott_included": false,
   "roaming": null
  }
 },
 {
  "plan_id": "P104",
  "plan_name": "Premium Max",
  "monthly_fee": 1199,
  "data_limit_gb": 100,
  "features": {
   "unlimited_calls": true,
   "ott_included": true,
   "roaming": "International"
  }
 }
]

Writing plans.json


In [2]:
from pyspark.sql import SparkSession
spark = SparkSession.builder.appName('join').getOrCreate()

In [3]:
from google.colab import files
uploaded = files.upload()

Saving customers.csv to customers.csv
Saving payments.csv to payments.csv
Saving usage.csv to usage.csv


In [4]:
customers_df = spark.read.csv(
    "customers.csv",
    header=True,
    inferSchema=True
)

usage_df = spark.read.csv(
    "usage.csv",
    header=True,
    inferSchema=True
)

payments_df = spark.read.csv(
    "payments.csv",
    header=True,
    inferSchema=True
)

plans_df = spark.read.option(
    "multiline",
    "true"
).json(
    "plans.json"
)

In [5]:
#1-6
customers_df.show()
usage_df.show()
payments_df.show()
plans_df.show(truncate=False)
customers_df.printSchema()
usage_df.printSchema()
payments_df.printSchema()
plans_df.printSchema()
print("Customers:", customers_df.count())
print("Usage:", usage_df.count())
print("Payments:", payments_df.count())
print("Plans:", plans_df.count())

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [6]:
#7
customers_df.write.mode(
    "overwrite"
).parquet(
    "bronze_customers"
)

usage_df.write.mode(
    "overwrite"
).parquet(
    "bronze_usage"
)

payments_df.write.mode(
    "overwrite"
).parquet(
    "bronze_payments"
)

plans_df.write.mode(
    "overwrite"
).parquet(
    "bronze_plans"
)

In [7]:
#8
customers_df.filter(
    customers_df.plan_id.isNull()
).show()

+-----------+-------------+---------+---------+---+------+-------+------+
|customer_id|customer_name|     city|    state|age|gender|plan_id|status|
+-----------+-------------+---------+---------+---+------+-------+------+
|        112|  Ayesha Khan|Hyderabad|Telangana| 28|Female|   NULL|Active|
+-----------+-------------+---------+---------+---+------+-------+------+



In [8]:
#9
usage_df.filter(
    usage_df.data_used_gb.isNull()
).show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1015|        105|2026-02-01 00:00:00|        NULL|        1450|      210|
+--------+-----------+-------------------+------------+------------+---------+



In [9]:
#10
payments_df.filter(
    payments_df.amount_paid.isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5011|        112|2026-01-01 00:00:00|       NULL|         UPI|       Success|
+----------+-----------+-------------------+-----------+------------+--------------+



In [10]:
#11
payments_df.filter(
    payments_df.payment_mode.isNull()
).show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5015|        105|2026-02-01 00:00:00|       1199|        NULL|       Pending|
+----------+-----------+-------------------+-----------+------------+--------------+



In [11]:
#12
customers_df = customers_df.na.fill(
    {"plan_id":"Unknown"}
)

customers_df.show()

+-----------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+-------------+---------+-----------+---+------+-------+--------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P103|Inactive|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|  Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|  Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|  Active|
|        107|  Arjun Verma|Hyderabad|  Telangana| 26|  Male|   P103|Inactive|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|  Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|  Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P

In [12]:
#13
usage_df = usage_df.na.fill(
    {"data_used_gb":0}
)

usage_df.show()

+--------+-----------+-------------------+------------+------------+---------+
|usage_id|customer_id|        usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-------------------+------------+------------+---------+
|    1001|        101|2026-01-01 00:00:00|          45|         900|      120|
|    1002|        102|2026-01-01 00:00:00|          30|         600|       80|
|    1003|        103|2026-01-01 00:00:00|          12|         250|       40|
|    1004|        104|2026-01-01 00:00:00|          55|        1100|      150|
|    1005|        105|2026-01-01 00:00:00|          75|        1500|      200|
|    1006|        106|2026-01-01 00:00:00|          28|         500|       60|
|    1007|        107|2026-01-01 00:00:00|          10|         200|       20|
|    1008|        108|2026-01-01 00:00:00|          80|        1600|      250|
|    1009|        109|2026-01-01 00:00:00|          48|         950|      100|
|    1010|        110|2026-01-01 00:00:00|          

In [13]:
#14
payments_df = payments_df.na.fill(
    {"amount_paid":0}
)

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [14]:
#15
payments_df = payments_df.na.fill(
    {"payment_mode":"Unknown"}
)

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|
+----------+-----------+-------------------+-----------+------------+--------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|
|      5007|        107|2026-01-01 00:00:00|        299|        Cash|       Pending|
|      5008|        108|2026-01-01 00:00:00|       1199|        Card|       Success|
|      5009|        109|2026-01-01 00:00:00|        499|         

In [15]:
#16
from pyspark.sql.functions import when,col

payments_df = payments_df.withColumn(
    "data_quality_status",
    when(
        (col("amount_paid")==0) |
        (col("payment_mode")=="Unknown"),
        "Incomplete"
    ).otherwise("Complete")
)

payments_df.show()

+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|payment_id|customer_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+----------+-----------+-------------------+-----------+------------+--------------+-------------------+
|      5001|        101|2026-01-01 00:00:00|        499|         UPI|       Success|           Complete|
|      5002|        102|2026-01-01 00:00:00|        799|        Card|       Success|           Complete|
|      5003|        103|2026-01-01 00:00:00|        299|        Cash|        Failed|           Complete|
|      5004|        104|2026-01-01 00:00:00|        499|         UPI|       Success|           Complete|
|      5005|        105|2026-01-01 00:00:00|       1199|        Card|       Success|           Complete|
|      5006|        106|2026-01-01 00:00:00|        799|         UPI|       Success|           Complete|
|      5007|        107|2026-01-01 00:00:00|        299

In [18]:
#17
payments_df.groupBy(
    "data_quality_status"
).count().show()

+-------------------+-----+
|data_quality_status|count|
+-------------------+-----+
|           Complete|   13|
|         Incomplete|    2|
+-------------------+-----+



In [17]:
customers_df.write.mode(
    "overwrite"
).parquet("silver_customers")

usage_df.write.mode(
    "overwrite"
).parquet("silver_usage")

payments_df.write.mode(
    "overwrite"
).parquet("silver_payments")

In [19]:
#18
plans_df.show(truncate=False)

+-------------+---------------------------+-----------+-------+------------+
|data_limit_gb|features                   |monthly_fee|plan_id|plan_name   |
+-------------+---------------------------+-----------+-------+------------+
|50           |{false, National, true}    |499        |P101   |Smart Basic |
|75           |{true, National, true}     |799        |P102   |Smart Plus  |
|25           |{false, NULL, false}       |299        |P103   |Budget Saver|
|100          |{true, International, true}|1199       |P104   |Premium Max |
+-------------+---------------------------+-----------+-------+------------+



In [20]:
#19
plans_df.printSchema()

root
 |-- data_limit_gb: long (nullable = true)
 |-- features: struct (nullable = true)
 |    |-- ott_included: boolean (nullable = true)
 |    |-- roaming: string (nullable = true)
 |    |-- unlimited_calls: boolean (nullable = true)
 |-- monthly_fee: long (nullable = true)
 |-- plan_id: string (nullable = true)
 |-- plan_name: string (nullable = true)



In [21]:
#20
from pyspark.sql.functions import col

plans_df.select(
    "plan_id",
    "plan_name",
    col("features.unlimited_calls").alias("unlimited_calls")
).show()

+-------+------------+---------------+
|plan_id|   plan_name|unlimited_calls|
+-------+------------+---------------+
|   P101| Smart Basic|           true|
|   P102|  Smart Plus|           true|
|   P103|Budget Saver|          false|
|   P104| Premium Max|           true|
+-------+------------+---------------+



In [22]:
#21
plans_df.select(
    "plan_id",
    "plan_name",
    col("features.ott_included").alias("ott_included")
).show()

+-------+------------+------------+
|plan_id|   plan_name|ott_included|
+-------+------------+------------+
|   P101| Smart Basic|       false|
|   P102|  Smart Plus|        true|
|   P103|Budget Saver|       false|
|   P104| Premium Max|        true|
+-------+------------+------------+



In [23]:
#22
plans_df.select(
    "plan_id",
    "plan_name",
    col("features.roaming").alias("roaming")
).show()

+-------+------------+-------------+
|plan_id|   plan_name|      roaming|
+-------+------------+-------------+
|   P101| Smart Basic|     National|
|   P102|  Smart Plus|     National|
|   P103|Budget Saver|         NULL|
|   P104| Premium Max|International|
+-------+------------+-------------+



In [24]:
#23
flat_plans_df = plans_df.select(
    "plan_id",
    "plan_name",
    "monthly_fee",
    "data_limit_gb",
    col("features.unlimited_calls").alias("unlimited_calls"),
    col("features.ott_included").alias("ott_included"),
    col("features.roaming").alias("roaming")
)

flat_plans_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|         NULL|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [25]:
#24
flat_plans_df.filter(
    flat_plans_df.roaming.isNull()
).show()

+-------+------------+-----------+-------------+---------------+------------+-------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|
+-------+------------+-----------+-------------+---------------+------------+-------+
|   P103|Budget Saver|        299|           25|          false|       false|   NULL|
+-------+------------+-----------+-------------+---------------+------------+-------+



In [26]:
#25
flat_plans_df = flat_plans_df.na.fill(
    {"roaming":"Not Available"}
)

flat_plans_df.show()

+-------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+------------+-----------+-------------+---------------+------------+-------------+
|   P101| Smart Basic|        499|           50|           true|       false|     National|
|   P102|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|Budget Saver|        299|           25|          false|       false|Not Available|
|   P104| Premium Max|       1199|          100|           true|        true|International|
+-------+------------+-----------+-------------+---------------+------------+-------------+



In [27]:
#26
flat_plans_df.agg(
    {"monthly_fee":"avg"}
).show()

+----------------+
|avg(monthly_fee)|
+----------------+
|           699.0|
+----------------+



In [28]:
#27
from pyspark.sql.functions import desc

flat_plans_df.orderBy(
    desc("monthly_fee")
).show(1)

+-------+-----------+-----------+-------------+---------------+------------+-------------+
|plan_id|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-----------+-------------+---------------+------------+-------------+
|   P104|Premium Max|       1199|          100|           true|        true|International|
+-------+-----------+-----------+-------------+---------------+------------+-------------+
only showing top 1 row


In [29]:
flat_plans_df.write.mode(
    "overwrite"
).parquet(
    "silver_plans"
)

In [30]:
#28
customer_plan_df = customers_df.join(
    flat_plans_df,
    on="plan_id",
    how="left"
)

customer_plan_df.show()

+-------+-----------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|
+-------+-----------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+
|   P101|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active| Smart Basic|        499|           50|           true|       false|     National|
|   P102|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|  Active|  Smart Plus|        799|           75|           true|        true|     National|
|   P103|        103|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|Inactive|Budget Saver|        299|           25|          false|       false|Not Available|
|   P101|        104|  Sneha Patel|  Che

In [31]:
#29
usage_customer_df = usage_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

usage_customer_df.show()

+-----------+--------+-------------------+------------+------------+---------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+--------+-------------------+------------+------------+---------+-------------+---------+-----------+---+------+-------+--------+
|        101|    1012|2026-02-01 00:00:00|          50|        1000|      130| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        101|    1001|2026-01-01 00:00:00|          45|         900|      120| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|    1013|2026-02-01 00:00:00|          34|         650|       85|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        102|    1002|2026-01-01 00:00:00|          30|         600|       80|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|

In [32]:
#30
payment_customer_df = payments_df.join(
    customers_df,
    on="customer_id",
    how="inner"
)

payment_customer_df.show()

+-----------+----------+-------------------+-----------+------------+--------------+-------------------+-------------+---------+-----------+---+------+-------+--------+
|customer_id|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|customer_name|     city|      state|age|gender|plan_id|  status|
+-----------+----------+-------------------+-----------+------------+--------------+-------------------+-------------+---------+-----------+---+------+-------+--------+
|        101|      5001|2026-01-01 00:00:00|        499|         UPI|       Success|           Complete| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|  Active|
|        102|      5002|2026-01-01 00:00:00|        799|        Card|       Success|           Complete|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|  Active|
|        103|      5003|2026-01-01 00:00:00|        299|        Cash|        Failed|           Complete|   Amit Kumar|   Mumbai|Maharashtra| 42|  Male|   P

In [33]:
#31
customer_usage_plan_df = customers_df.join(
    flat_plans_df,
    on="plan_id",
    how="left"
).join(
    usage_df,
    on="customer_id",
    how="left"
)

customer_usage_plan_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+
|        101|   P101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active| Smart Basic|        499|           50|           true|       false|     National|    1012|2026-02-01 00:00:00|          50|        1000|      130|
|        101|   P101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|  Active| Smart Basic|        49

In [34]:
#32
telecom_master_df = customers_df.join(
    flat_plans_df,
    on="plan_id",
    how="left"
).join(
    usage_df,
    on="customer_id",
    how="left"
).join(
    payments_df,
    on="customer_id",
    how="left"
)

telecom_master_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101|   P101| Rahul Sharma|Hyderabad|  Telan

In [37]:
#33
customers_df.join(
    flat_plans_df,
    on="plan_id",
    how="left"
).filter(
    flat_plans_df.plan_name.isNull()
).show()

+-------+-----------+-------------+---------+-----------+---+------+------+---------+-----------+-------------+---------------+------------+-------+
|plan_id|customer_id|customer_name|     city|      state|age|gender|status|plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|
+-------+-----------+-------------+---------+-----------+---+------+------+---------+-----------+-------------+---------------+------------+-------+
|   P105|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|Active|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
|Unknown|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Active|     NULL|       NULL|         NULL|           NULL|        NULL|   NULL|
+-------+-----------+-------------+---------+-----------+---+------+------+---------+-----------+-------------+---------------+------------+-------+



In [36]:
#34
usage_df.join(
    customers_df,
    on="customer_id",
    how="left"
).filter(
    customers_df.customer_name.isNull()
).show()

+-----------+--------+-------------------+------------+------------+---------+-------------+----+-----+----+------+-------+------+
|customer_id|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|customer_name|city|state| age|gender|plan_id|status|
+-----------+--------+-------------------+------------+------------+---------+-------------+----+-----+----+------+-------+------+
|        120|    1011|2026-01-01 00:00:00|          60|        1300|      140|         NULL|NULL| NULL|NULL|  NULL|   NULL|  NULL|
+-----------+--------+-------------------+------------+------------+---------+-------------+----+-----+----+------+-------+------+



In [38]:
#35
payments_df.join(
    customers_df,
    on="customer_id",
    how="left"
).filter(
    customers_df.customer_name.isNull()
).show()

+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+
|customer_id|payment_id|bill_month|amount_paid|payment_mode|payment_status|data_quality_status|customer_name|city|state|age|gender|plan_id|status|
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+
+-----------+----------+----------+-----------+------------+--------------+-------------------+-------------+----+-----+---+------+-------+------+



In [39]:
#36
telecom_master_df.count()

24

In [40]:
#37
telecom_master_df.select(
    "customer_name",
    "plan_name",
    "monthly_fee",
    "data_limit_gb"
).show()

+-------------+------------+-----------+-------------+
|customer_name|   plan_name|monthly_fee|data_limit_gb|
+-------------+------------+-----------+-------------+
| Rahul Sharma| Smart Basic|        499|           50|
| Rahul Sharma| Smart Basic|        499|           50|
| Rahul Sharma| Smart Basic|        499|           50|
| Rahul Sharma| Smart Basic|        499|           50|
|  Priya Reddy|  Smart Plus|        799|           75|
|  Priya Reddy|  Smart Plus|        799|           75|
|  Priya Reddy|  Smart Plus|        799|           75|
|  Priya Reddy|  Smart Plus|        799|           75|
|   Amit Kumar|Budget Saver|        299|           25|
|  Sneha Patel| Smart Basic|        499|           50|
|  Sneha Patel| Smart Basic|        499|           50|
|  Sneha Patel| Smart Basic|        499|           50|
|  Sneha Patel| Smart Basic|        499|           50|
|   Farhan Ali| Premium Max|       1199|          100|
|   Farhan Ali| Premium Max|       1199|          100|
|   Farhan

In [41]:
#38
telecom_master_df.select(
    "customer_name",
    "usage_month",
    "data_used_gb",
    "amount_paid"
).show()

+-------------+-------------------+------------+-----------+
|customer_name|        usage_month|data_used_gb|amount_paid|
+-------------+-------------------+------------+-----------+
| Rahul Sharma|2026-02-01 00:00:00|          50|        499|
| Rahul Sharma|2026-02-01 00:00:00|          50|        499|
| Rahul Sharma|2026-01-01 00:00:00|          45|        499|
| Rahul Sharma|2026-01-01 00:00:00|          45|        499|
|  Priya Reddy|2026-02-01 00:00:00|          34|        799|
|  Priya Reddy|2026-02-01 00:00:00|          34|        799|
|  Priya Reddy|2026-01-01 00:00:00|          30|        799|
|  Priya Reddy|2026-01-01 00:00:00|          30|        799|
|   Amit Kumar|2026-01-01 00:00:00|          12|        299|
|  Sneha Patel|2026-02-01 00:00:00|          58|        499|
|  Sneha Patel|2026-02-01 00:00:00|          58|        499|
|  Sneha Patel|2026-01-01 00:00:00|          55|        499|
|  Sneha Patel|2026-01-01 00:00:00|          55|        499|
|   Farhan Ali|2026-02-0

In [42]:
#39
telecom_master_df.filter(
    telecom_master_df.status == "Active"
).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|
+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+
|        101|   P101| Rahul Sharma|Hyderabad|  Telangana| 35|

In [43]:
#40
telecom_master_df.write.mode(
    "overwrite"
).parquet(
    "silver_telecom_master"
)

In [44]:
#41
telecom_master_df.write.mode(
    "overwrite"
).parquet(
    "silver_telecom_master"
)

In [56]:
#42
telecom_master_df.groupBy(
    "churn_risk"
).count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [46]:
#43
telecom_master_df = telecom_master_df.withColumn(
    "payment_category",
    when(col("amount_paid") >= 1000,"High Payment")
    .when(col("amount_paid") >= 500,"Medium Payment")
    .otherwise("Low Payment")
)

telecom_master_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+
|

In [47]:
#44
telecom_master_df.groupBy(
    "payment_category"
).count().show()

+----------------+-----+
|payment_category|count|
+----------------+-----+
|     Low Payment|   13|
|  Medium Payment|    6|
|    High Payment|    5|
+----------------+-----+



In [48]:
#45
telecom_master_df = telecom_master_df.withColumn(
    "churn_risk",
    when(
        (col("status")=="Inactive") |
        (col("payment_status")!="Success"),
        "High Risk"
    )
    .when(
        col("data_used_gb") < 15,
        "Medium Risk"
    )
    .otherwise("Low Risk")
)

telecom_master_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+---------------

In [49]:
#46
telecom_master_df.groupBy(
    "churn_risk"
).count().show()

+-----------+-----+
| churn_risk|count|
+-----------+-----+
|   Low Risk|   19|
|Medium Risk|    1|
|  High Risk|    4|
+-----------+-----+



In [50]:
#47
telecom_master_df = telecom_master_df.withColumn(
    "over_usage_gb",
    col("data_used_gb") - col("data_limit_gb")
)

telecom_master_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--

In [51]:
#48
telecom_master_df = telecom_master_df.withColumn(
    "over_usage_flag",
    when(
        col("over_usage_gb") > 0,
        "Yes"
    ).otherwise("No")
)

telecom_master_df.show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+---------------

In [52]:
#49
telecom_master_df.groupBy(
    "over_usage_flag"
).count().show()

+---------------+-----+
|over_usage_flag|count|
+---------------+-----+
|             No|   20|
|            Yes|    4|
+---------------+-----+



In [57]:
#50
from pyspark.sql.functions import when, col

telecom_master_df = telecom_master_df.withColumn(
    "usage_category",
    when(col("data_used_gb") >= 70, "Heavy User")
    .when(col("data_used_gb") >= 30, "Medium User")
    .otherwise("Light User")
)

telecom_master_df.filter(
    telecom_master_df.usage_category == "Heavy User"
).show()

+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+---

In [58]:
#51
telecom_master_df.filter(
    telecom_master_df.churn_risk == "High Risk"
).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+--------

In [59]:
#52
telecom_master_df.filter(
    telecom_master_df.over_usage_flag == "Yes"
).show()

+-----------+-------+-------------+-------+----------+---+------+------+-----------+-----------+-------------+---------------+------------+--------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name|   city|     state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included| roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+-------+----------+---+------+------+-----------+-----------+-------------+---------------+------------+--------+--------+-------------------+------------+------------+---------+----------+-------------------+

In [60]:
#53
telecom_master_df.filter(
    telecom_master_df.plan_name == "Premium Max"
).show()

+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-

In [61]:
#54
telecom_master_df.filter(
    telecom_master_df.ott_included == True
).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+-----

In [62]:
#55
telecom_master_df.write.mode(
    "overwrite"
).parquet(
    "gold_telecom_transformed"
)

In [63]:
#56
telecom_master_df.groupBy(
    "city"
).count().show()

+---------+-----+
|     city|count|
+---------+-----+
|Bangalore|    5|
|    Kochi|    1|
|  Chennai|    4|
|   Mumbai|    2|
|     Pune|    1|
|    Delhi|    5|
|Hyderabad|    6|
+---------+-----+



In [64]:
#57
telecom_master_df.groupBy(
    "state"
).count().show()

+-----------+-----+
|      state|count|
+-----------+-----+
|  Karnataka|    5|
|     Kerala|    1|
| Tamil Nadu|    4|
|      Delhi|    5|
|  Telangana|    6|
|Maharashtra|    3|
+-----------+-----+



In [65]:
#58
telecom_master_df.groupBy(
    "plan_name"
).count().show()

+------------+-----+
|   plan_name|count|
+------------+-----+
|        NULL|    2|
| Smart Basic|    9|
|Budget Saver|    2|
| Premium Max|    5|
|  Smart Plus|    6|
+------------+-----+



In [66]:
#59
from pyspark.sql.functions import sum

telecom_master_df.groupBy(
    "plan_name"
).agg(
    sum("data_used_gb").alias("total_data_usage")
).show()

+------------+----------------+
|   plan_name|total_data_usage|
+------------+----------------+
|        NULL|            NULL|
| Smart Basic|             464|
|Budget Saver|              22|
| Premium Max|             230|
|  Smart Plus|             188|
+------------+----------------+



In [67]:
#60
from pyspark.sql.functions import avg

telecom_master_df.groupBy(
    "plan_name"
).agg(
    avg("data_used_gb").alias("avg_data_usage")
).show()

+------------+------------------+
|   plan_name|    avg_data_usage|
+------------+------------------+
|        NULL|              NULL|
| Smart Basic| 51.55555555555556|
|Budget Saver|              11.0|
| Premium Max|              46.0|
|  Smart Plus|31.333333333333332|
+------------+------------------+



In [68]:
#61
telecom_master_df.groupBy(
    "city"
).agg(
    sum("call_minutes").alias("total_call_minutes")
).show()

+---------+------------------+
|     city|total_call_minutes|
+---------+------------------+
|Bangalore|              3450|
|    Kochi|              1600|
|  Chennai|              4600|
|   Mumbai|               250|
|     Pune|               500|
|    Delhi|              6600|
|Hyderabad|              4000|
+---------+------------------+



In [69]:
#62
telecom_master_df.groupBy(
    "state"
).agg(
    sum("sms_count").alias("total_sms")
).show()


+-----------+---------+
|      state|total_sms|
+-----------+---------+
|  Karnataka|      430|
|     Kerala|      250|
| Tamil Nadu|      620|
|      Delhi|      910|
|  Telangana|      520|
|Maharashtra|      100|
+-----------+---------+



In [70]:
#63
telecom_master_df.filter(
    telecom_master_df.payment_status == "Success"
).agg(
    sum("amount_paid").alias("total_revenue")
).show()

+-------------+
|total_revenue|
+-------------+
|        12882|
+-------------+



In [71]:
#64
telecom_master_df.groupBy(
    "city"
).agg(
    sum("amount_paid").alias("total_revenue")
).show()

+---------+-------------+
|     city|total_revenue|
+---------+-------------+
|Bangalore|         3695|
|    Kochi|         1199|
|  Chennai|         1996|
|   Mumbai|          299|
|     Pune|          799|
|    Delhi|         5595|
|Hyderabad|         2295|
+---------+-------------+



In [72]:
#65
telecom_master_df.groupBy(
    "plan_name"
).agg(
    sum("amount_paid").alias("total_revenue")
).show()

+------------+-------------+
|   plan_name|total_revenue|
+------------+-------------+
|        NULL|            0|
| Smart Basic|         4491|
|Budget Saver|          598|
| Premium Max|         5995|
|  Smart Plus|         4794|
+------------+-------------+



In [73]:
#66
from pyspark.sql.window import Window
from pyspark.sql.functions import rank

usage_window = Window.orderBy(
    telecom_master_df.data_used_gb.desc()
)

telecom_master_df.withColumn(
    "usage_rank",
    rank().over(usage_window)
).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+----------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|usage_rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------

In [74]:
#67
payment_window = Window.orderBy(
    telecom_master_df.amount_paid.desc()
)

telecom_master_df.withColumn(
    "payment_rank",
    rank().over(payment_window)
).show()

+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+--------------+------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|usage_category|payment_rank|
+-----------+-------+-------------+---------+-----------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+--

In [75]:
#68
telecom_master_df.withColumn(
    "usage_rank",
    rank().over(usage_window)
).filter(
    "usage_rank <= 3"
).show()

+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+----------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|usage_rank|
+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-

In [76]:
#69
telecom_master_df.withColumn(
    "payment_rank",
    rank().over(payment_window)
).filter(
    "payment_rank <= 3"
).show()

+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+--------------+------------+
|customer_id|plan_id|customer_name| city| state|age|gender|status|  plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|usage_category|payment_rank|
+-----------+-------+-------------+-----+------+---+------+------+-----------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+------

In [77]:
#70
city_window = Window.partitionBy(
    "city"
).orderBy(
    telecom_master_df.amount_paid.desc()
)

telecom_master_df.withColumn(
    "city_rank",
    rank().over(city_window)
).filter(
    "city_rank = 1"
).show()

+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+-----------+-------------+---------------+--------------+---------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|  status|   plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|      roaming|usage_id|        usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category| churn_risk|over_usage_gb|over_usage_flag|usage_category|city_rank|
+-----------+-------+-------------+---------+-----------+---+------+--------+------------+-----------+-------------+---------------+------------+-------------+--------+-------------------+------------

In [78]:
#71
customers_df.createOrReplaceTempView("customers")

usage_df.createOrReplaceTempView("usage")

payments_df.createOrReplaceTempView("payments")

flat_plans_df.createOrReplaceTempView("plans")

In [79]:
#72
spark.sql("""
SELECT *
FROM customers
WHERE status='Active'
""").show()

+-----------+-------------+---------+-----------+---+------+-------+------+
|customer_id|customer_name|     city|      state|age|gender|plan_id|status|
+-----------+-------------+---------+-----------+---+------+-------+------+
|        101| Rahul Sharma|Hyderabad|  Telangana| 35|  Male|   P101|Active|
|        102|  Priya Reddy|Bangalore|  Karnataka| 29|Female|   P102|Active|
|        104|  Sneha Patel|  Chennai| Tamil Nadu| 31|Female|   P101|Active|
|        105|   Farhan Ali|    Delhi|      Delhi| 55|  Male|   P104|Active|
|        106|   Neha Singh|     Pune|Maharashtra| 38|Female|   P102|Active|
|        108|   Meera Nair|    Kochi|     Kerala| 48|Female|   P104|Active|
|        109|    Kiran Rao|Bangalore|  Karnataka| 33|  Male|   P101|Active|
|        110|  Nisha Reddy|    Delhi|      Delhi| 41|Female|   P102|Active|
|        111|   Ravi Kumar|   Mumbai|Maharashtra| 45|  Male|   P105|Active|
|        112|  Ayesha Khan|Hyderabad|  Telangana| 28|Female|Unknown|Active|
+-----------

In [80]:
#73
spark.sql("""
SELECT city,
       COUNT(*) AS customer_count
FROM customers
GROUP BY city
""").show()

+---------+--------------+
|     city|customer_count|
+---------+--------------+
|Bangalore|             2|
|    Kochi|             1|
|  Chennai|             1|
|   Mumbai|             2|
|     Pune|             1|
|    Delhi|             2|
|Hyderabad|             3|
+---------+--------------+



In [81]:
#74
telecom_master_df.createOrReplaceTempView(
    "telecom_master"
)

spark.sql("""
SELECT plan_name,
       SUM(amount_paid) AS revenue
FROM telecom_master
GROUP BY plan_name
ORDER BY revenue DESC
""").show()

+------------+-------+
|   plan_name|revenue|
+------------+-------+
| Premium Max|   5995|
|  Smart Plus|   4794|
| Smart Basic|   4491|
|Budget Saver|    598|
|        NULL|      0|
+------------+-------+



In [82]:
#75
spark.sql("""
SELECT customer_id,
       customer_name,
       data_used_gb
FROM telecom_master
WHERE usage_category='Heavy User'
""").show()

+-----------+-------------+------------+
|customer_id|customer_name|data_used_gb|
+-----------+-------------+------------+
|        105|   Farhan Ali|          75|
|        105|   Farhan Ali|          75|
|        108|   Meera Nair|          80|
+-----------+-------------+------------+



In [83]:
#76
spark.sql("""
SELECT customer_id,
       customer_name,
       churn_risk
FROM telecom_master
WHERE churn_risk='High Risk'
""").show()

+-----------+-------------+----------+
|customer_id|customer_name|churn_risk|
+-----------+-------------+----------+
|        103|   Amit Kumar| High Risk|
|        105|   Farhan Ali| High Risk|
|        105|   Farhan Ali| High Risk|
|        107|  Arjun Verma| High Risk|
+-----------+-------------+----------+



In [84]:
#77
spark.sql("""
SELECT *
FROM telecom_master
WHERE plan_name IS NULL
""").show()


+-----------+-------+-------------+---------+-----------+---+------+------+---------+-----------+-------------+---------------+------------+-------+--------+-----------+------------+------------+---------+----------+-------------------+-----------+------------+--------------+-------------------+----------------+----------+-------------+---------------+--------------+
|customer_id|plan_id|customer_name|     city|      state|age|gender|status|plan_name|monthly_fee|data_limit_gb|unlimited_calls|ott_included|roaming|usage_id|usage_month|data_used_gb|call_minutes|sms_count|payment_id|         bill_month|amount_paid|payment_mode|payment_status|data_quality_status|payment_category|churn_risk|over_usage_gb|over_usage_flag|usage_category|
+-----------+-------+-------------+---------+-----------+---+------+------+---------+-----------+-------------+---------------+------------+-------+--------+-----------+------------+------------+---------+----------+-------------------+-----------+------------

In [85]:
#78
spark.sql("""
SELECT customer_id,
       customer_name,
       payment_status
FROM telecom_master
WHERE payment_status IN ('Failed','Pending')
""").show()


+-----------+-------------+--------------+
|customer_id|customer_name|payment_status|
+-----------+-------------+--------------+
|        103|   Amit Kumar|        Failed|
|        105|   Farhan Ali|       Pending|
|        105|   Farhan Ali|       Pending|
|        107|  Arjun Verma|       Pending|
+-----------+-------------+--------------+



In [86]:
march_usage = [
(1016,101,"2026-03",52,1050,140),
(1017,102,"2026-03",36,700,90),
(1018,105,"2026-03",82,1700,260)
]

incremental_usage_df = spark.createDataFrame(
    march_usage,
    [
        "usage_id",
        "customer_id",
        "usage_month",
        "data_used_gb",
        "call_minutes",
        "sms_count"
    ]
)

incremental_usage_df.show()

+--------+-----------+-----------+------------+------------+---------+
|usage_id|customer_id|usage_month|data_used_gb|call_minutes|sms_count|
+--------+-----------+-----------+------------+------------+---------+
|    1016|        101|    2026-03|          52|        1050|      140|
|    1017|        102|    2026-03|          36|         700|       90|
|    1018|        105|    2026-03|          82|        1700|      260|
+--------+-----------+-----------+------------+------------+---------+



In [87]:
incremental_usage_df.write.mode(
    "append"
).parquet(
    "silver_usage"
)

In [88]:
updated_usage_df = spark.read.parquet(
    "silver_usage"
)

print(
    "Original Count:",
    usage_df.count()
)

print(
    "Updated Count:",
    updated_usage_df.count()
)

Original Count: 15
Updated Count: 18


In [89]:
#79
telecom_master_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "usage_month",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_flag",
    "amount_paid",
    "payment_status",
    "churn_risk"
).show()


+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|customer_id|customer_name|     city|   plan_name|        usage_month|data_used_gb|data_limit_gb|over_usage_flag|amount_paid|payment_status| churn_risk|
+-----------+-------------+---------+------------+-------------------+------------+-------------+---------------+-----------+--------------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-02-01 00:00:00|          50|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45|           50|             No|        499|       Success|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|2026-01-01 00:00:00|          45

In [90]:
#80
from pyspark.sql.functions import sum,avg,count

telecom_master_df.groupBy(
    "plan_name"
).agg(
    count("customer_id").alias("total_customers"),
    sum("data_used_gb").alias("total_data_usage"),
    avg("data_used_gb").alias("average_data_usage"),
    sum("amount_paid").alias("total_revenue")
).show()

+------------+---------------+----------------+------------------+-------------+
|   plan_name|total_customers|total_data_usage|average_data_usage|total_revenue|
+------------+---------------+----------------+------------------+-------------+
|        NULL|              2|            NULL|              NULL|            0|
| Smart Basic|              9|             464| 51.55555555555556|         4491|
|Budget Saver|              2|              22|              11.0|          598|
| Premium Max|              5|             230|              46.0|         5995|
|  Smart Plus|              6|             188|31.333333333333332|         4794|
+------------+---------------+----------------+------------------+-------------+



In [91]:
#81
telecom_master_df.groupBy(
    "city"
).agg(
    count("customer_id").alias("total_customers"),
    sum("amount_paid").alias("total_revenue"),
    avg("amount_paid").alias("average_payment")
).show()

+---------+---------------+-------------+---------------+
|     city|total_customers|total_revenue|average_payment|
+---------+---------------+-------------+---------------+
|Bangalore|              5|         3695|          739.0|
|    Kochi|              1|         1199|         1199.0|
|  Chennai|              4|         1996|          499.0|
|   Mumbai|              2|          299|          299.0|
|     Pune|              1|          799|          799.0|
|    Delhi|              5|         5595|         1119.0|
|Hyderabad|              6|         2295|          382.5|
+---------+---------------+-------------+---------------+



In [92]:
#82
telecom_master_df.select(
    "customer_id",
    "customer_name",
    "city",
    "plan_name",
    "payment_status",
    "status",
    "churn_risk"
).show()

+-----------+-------------+---------+------------+--------------+--------+-----------+
|customer_id|customer_name|     city|   plan_name|payment_status|  status| churn_risk|
+-----------+-------------+---------+------------+--------------+--------+-----------+
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        101| Rahul Sharma|Hyderabad| Smart Basic|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        102|  Priya Reddy|Bangalore|  Smart Plus|       Success|  Active|   Low Risk|
|        103|   Amit Kumar|   Mumbai|Budget

In [93]:
#83
telecom_master_df.filter(
    telecom_master_df.over_usage_flag == "Yes"
).select(
    "customer_id",
    "customer_name",
    "plan_name",
    "data_used_gb",
    "data_limit_gb",
    "over_usage_gb"
).show()

+-----------+-------------+-----------+------------+-------------+-------------+
|customer_id|customer_name|  plan_name|data_used_gb|data_limit_gb|over_usage_gb|
+-----------+-------------+-----------+------------+-------------+-------------+
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          58|           50|            8|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
|        104|  Sneha Patel|Smart Basic|          55|           50|            5|
+-----------+-------------+-----------+------------+-------------+-------------+

